# Research-tier video — the free half of the video corpus

This notebook animates corpus stills on a **Colab free GPU** and writes the clips plus a
manifest that `adproviders.PrerenderedVideoProvider` reads back. It is the `$0` line in
the budget, and it is what makes a video-stage evaluation possible at all.

The arithmetic that forces it: the premium tier is Kling at `$0.07/s`, so a 10 s clip is
`$0.70` and the `$20` premium allocation buys about **28 videos** — enough for golden
demos, nowhere near enough to validate a ranker. Open weights cost nothing per clip.

**The boundary is the same one `colab_embeddings.ipynb` uses.** Torch lives here; the
laptop reads files. Nothing on the laptop imports torch, needs a GPU, or calls a network
API, so the same pipeline code paths run in both places.

## What you need

1. `fixtures/corpus/manifest.json` and the corpus images, from
   `scripts/extract_features.py` (which writes the manifest as its Colab handoff).
2. A T4 runtime: **Runtime > Change runtime type > T4 GPU**.

## What comes back

A zip holding `research/clips.json` and `research/*.mp4`. Unzip it into the storage root
(`fixtures/`), then:

```
AD_VIDEO_PROVIDER=ltx-video .venv/bin/python scripts/generate_corpus.py --sets 20 --videos
```

No `--confirm-spend` is needed and mock mode does not have to be turned off: the research
tier is on the registry's free-provider allowlist, because requiring `AD_PROVIDER_MODE=live`
to run the *free* corpus would also arm the paid image provider.

## 1. Environment

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "no GPU — set Runtime > T4"
# ffmpeg is preinstalled on Colab; checked rather than assumed, because every clip
# written here goes through it and a missing binary should fail now, not in cell 6.
!ffmpeg -version | head -1

In [ ]:
# diffusers for LTX-Video. Pinned loosely: the pipeline class names have been stable,
# and pinning hard against a Colab image that moves under you is its own failure mode.
!pip install -q "diffusers>=0.32" "transformers>=4.44" "accelerate>=0.34" imageio imageio-ffmpeg

## 2. Upload the corpus

In [ ]:
import json
import zipfile
from pathlib import Path

from google.colab import files

WORK = Path("/content/work")
WORK.mkdir(exist_ok=True)

print("Upload corpus.zip (cd fixtures && zip -qr /tmp/corpus.zip corpus/) and manifest.json")
uploaded = files.upload()

for name in uploaded:
    if name.endswith(".zip"):
        with zipfile.ZipFile(name) as zf:
            zf.extractall(WORK)
        print(f"extracted {name}: {len(zf.namelist())} entries")
    elif name.endswith(".json"):
        Path(WORK / "manifest.json").write_bytes(uploaded[name])
        print(f"took {name} as the manifest")

MANIFEST = json.loads((WORK / "manifest.json").read_text())
print(f"manifest lists {len(MANIFEST['items'])} items")

## 3. Which stills to animate

Not all of them. A 9 s clip at 480p takes roughly 3-5 minutes on a T4, so 60 clips is
around four hours — longer than one free Colab session reliably lasts. So the batch is
bounded, ordered deterministically, and **resumable**: a clip already in the output
directory is skipped, and re-running after a disconnect continues rather than restarts.

The plan's target is ~60 research-tier videos. Expect two or three sessions.

In [ ]:
# Ordered by item id so two sessions agree on what "the first 30" means. A random
# sample would re-draw on every reconnect and the resume logic would be useless.
ITEMS = sorted(
    (it for it in MANIFEST["items"] if not it.get("is_decoy")),
    key=lambda it: it["item_id"],
)
BATCH_START = 0  # advance this between sessions
BATCH_SIZE = 30
BATCH = ITEMS[BATCH_START : BATCH_START + BATCH_SIZE]

DURATION_S = 9.0  # the project's default; inside the 8-10 s commitment
FPS = 24

print(f"{len(BATCH)} items this session (of {len(ITEMS)} total)")
print(f"  {BATCH_START} .. {BATCH_START + len(BATCH) - 1}")
for it in BATCH[:3]:
    print(f"  {it['item_id']}  {it['key']}")

## 4. The fingerprint has to match what the laptop will ask for

This is the one place where a silent mismatch would waste the whole session: every lookup
would miss and the provider would refuse — correctly, but after four hours of GPU time.

`PrerenderedVideoProvider` finds a clip by fingerprinting **`(start-frame sha256, motion
intent, duration)`**.

Note what is *not* in that key: the motion **prompt text**. That would be the obvious
choice, since it is what the video model actually receives. It cannot be —
`adworker.briefs.compile_briefs` expands briefs through an LLM, so the prompt wording is
not reproducible between runs, and a fingerprint built on it would break the moment the
compiler phrased the same design point differently. `MotionIntent` is a short stable enum
that both sides have: read from the manifest here, read off `brief.design_point.motion`
in the pipeline.

The prompt fed to LTX-Video below is therefore built from the intent, and its wording is
free to change without invalidating a single clip.

In [ ]:
import hashlib


def clip_fingerprint(start_digest: str, motion_intent: str, duration_seconds: float) -> str:
    """Byte-for-byte the same function as adproviders.prerendered.clip_fingerprint.

    Duplicated rather than imported because this notebook cannot import the repo.
    The two copies agreeing is asserted on the laptop side by
    tests/test_prerendered.py::test_the_notebook_fingerprint_matches_the_provider,
    which pins the exact payload shape — so a drift between them fails a test rather
    than silently missing every lookup after a GPU session.
    """
    payload = json.dumps(
        {
            "start": start_digest,
            "motion": str(motion_intent).strip(),
            "duration": round(float(duration_seconds), 2),
        },
        sort_keys=True,
    )
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:32]


def sha256_of(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


# Wording is free to change: it is not part of the fingerprint. Keep the intent
# vocabulary in step with adschema.MotionIntent, which is what *is* fingerprinted.
MOTION_PROMPTS = {
    "slow_dolly_in": "slow cinematic dolly in toward the subject, product held steady",
    "slow_dolly_out": "slow cinematic dolly out, revealing the scene around the subject",
    "orbit_left": "smooth camera orbit to the left around the subject",
    "product_present": "the model turns the product toward the camera and presents it",
    "handheld_drift": "subtle handheld drift, natural breathing motion, gentle hair movement",
    "static_subtle": "near-static shot, subtle fabric and hair movement only",
}

missing = [it["item_id"] for it in BATCH if not it.get("motion")]
if missing:
    raise SystemExit(
        f"{len(missing)} items have no motion intent in the manifest, so their clips "
        f"could never be looked up: {missing[:5]}. Re-run scripts/extract_features.py."
    )
unknown = sorted({it["motion"] for it in BATCH} - set(MOTION_PROMPTS))
if unknown:
    raise SystemExit(
        f"no prompt for motion intent(s) {unknown}. MotionIntent has gained a member; "
        "add it to MOTION_PROMPTS above."
    )
print("every item in the batch carries a known motion intent")

## 5. Load LTX-Video

In [ ]:
import torch
from diffusers import LTXImageToVideoPipeline

MODEL_ID = "Lightricks/LTX-Video"
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

pipe = LTXImageToVideoPipeline.from_pretrained(MODEL_ID, torch_dtype=DTYPE)
pipe.to("cuda")
# Slices attention rather than holding the whole map: a T4 has 16 GB and this is the
# difference between running and an out-of-memory halt partway through the batch.
pipe.enable_attention_slicing()
pipe.vae.enable_tiling()
print(f"loaded {MODEL_ID} in {DTYPE}")

## 6. Generate

LTX-Video wants a frame count of the form `8k + 1`, and its own rate is 24 fps. A 9 s clip
is therefore 217 frames (`8 x 27 + 1`), which lands at 9.04 s — inside the 8-10 s window,
and the laptop measures the real duration off the file rather than trusting this number.

Each clip is written and its manifest entry appended **immediately**. A session that dies
at clip 22 leaves 22 usable clips and a valid manifest, rather than nothing.

In [ ]:
import time

import imageio.v3 as iio
import numpy as np

OUT = WORK / "research"
OUT.mkdir(exist_ok=True)
CLIPS_JSON = OUT / "clips.json"


def frames_for(duration_s: float, fps: int) -> int:
    """LTX-Video accepts 8k+1 frames only; round to the nearest valid count."""
    raw = duration_s * fps
    k = max(1, round((raw - 1) / 8))
    return 8 * k + 1


def load_existing() -> dict:
    if CLIPS_JSON.is_file():
        return json.loads(CLIPS_JSON.read_text())
    return {"version": 1, "notebook": "colab_video.ipynb", "clips": []}


record = load_existing()
done = {c["fingerprint"] for c in record["clips"]}
print(f"{len(done)} clips already present; resuming")

N_FRAMES = frames_for(DURATION_S, FPS)
print(f"{N_FRAMES} frames at {FPS} fps = {N_FRAMES / FPS:.2f}s per clip")

In [ ]:
from PIL import Image

started_all = time.time()
for index, item in enumerate(BATCH, start=1):
    source = WORK / item["key"]
    if not source.is_file():
        print(f"  [{index}/{len(BATCH)}] {item['item_id']}: SOURCE MISSING at {item['key']}")
        continue

    digest = sha256_of(source)
    fingerprint = clip_fingerprint(digest, item["motion"], DURATION_S)
    if fingerprint in done:
        print(f"  [{index}/{len(BATCH)}] {item['item_id']}: already done")
        continue

    image = Image.open(source).convert("RGB")
    # Downscaled to a 480 px long edge, and to a multiple of 32. 480p is enough for
    # ranking research and is what keeps a clip inside a few minutes on a T4.
    long_edge = 480
    scale = long_edge / max(image.size)
    width = max(32, int(round(image.width * scale / 32)) * 32)
    height = max(32, int(round(image.height * scale / 32)) * 32)
    image = image.resize((width, height), Image.LANCZOS)

    seed = int(item.get("seed") or 0)
    generator = torch.Generator(device="cuda").manual_seed(seed)

    started = time.time()
    result = pipe(
        image=image,
        prompt=MOTION_PROMPTS[item["motion"]],
        negative_prompt="worst quality, distorted, jittery, watermark, text",
        width=width,
        height=height,
        num_frames=N_FRAMES,
        num_inference_steps=40,
        generator=generator,
    )
    elapsed = time.time() - started

    frames = [np.asarray(f) for f in result.frames[0]]
    out_key = f"research/{item['item_id']}.mp4"
    iio.imwrite(
        WORK / out_key,
        np.stack(frames),
        fps=FPS,
        codec="libx264",
        pixelformat="yuv420p",
    )

    record["clips"].append(
        {
            "fingerprint": fingerprint,
            "key": out_key,
            "model": "ltx-video",
            "duration_seconds": len(frames) / FPS,
            "fps": FPS,
            "was_chained": False,
            "seed": seed,
            # LTX-Video takes a torch generator, so the seed is genuinely honoured —
            # unlike Kling, whose image-to-video endpoint has no seed at all. This is
            # the research tier's one real advantage for ablation work.
            "seed_honoured": True,
            "note": f"{width}x{height}, {N_FRAMES} frames, 40 steps, {item['motion']}",
        }
    )
    done.add(fingerprint)
    # Written after every clip, not at the end: a disconnect at clip 22 should leave
    # 22 usable clips and a valid manifest.
    CLIPS_JSON.write_text(json.dumps(record, indent=2))
    print(
        f"  [{index}/{len(BATCH)}] {item['item_id']}: {len(frames)} frames "
        f"in {elapsed:.0f}s -> {out_key}"
    )

minutes = (time.time() - started_all) / 60
print(f"\n{len(record['clips'])} clips total, session took {minutes:.1f} min")

## 7. Verify before downloading

Two checks, both of which have caught something in this project before.

**Durations are read back from the files**, not from the numbers written into the manifest
— those came from a frame count, and an encoder that drops trailing frames would make them
a fiction.

**Clips must actually move.** A pipeline that silently produced a still would write a
perfectly valid MP4 of a frozen frame, and every motion feature would read zero. Mean
absolute frame difference separates the two immediately.

In [ ]:
import subprocess

bad = []
for entry in record["clips"]:
    path = WORK / entry["key"]
    probe = subprocess.run(
        [
            "ffprobe",
            "-v",
            "error",
            "-show_entries",
            "format=duration",
            "-of",
            "csv=p=0",
            str(path),
        ],
        capture_output=True,
        text=True,
    )
    measured = float(probe.stdout.strip() or 0.0)

    clip = iio.imread(path)
    grays = clip.mean(axis=-1) / 255.0
    motion = float(np.abs(np.diff(grays, axis=0)).mean())

    ok = 8.0 <= measured <= 10.0 and motion > 1e-3
    if not ok:
        bad.append((entry["key"], measured, motion))
    print(f"  {entry['key']}: {measured:.2f}s  motion={motion:.4f}  {'ok' if ok else 'FAIL'}")

print()
if bad:
    print(f"{len(bad)} clips failed verification and should not be used:")
    for key, measured, motion in bad:
        reason = "outside 8-10s" if not 8.0 <= measured <= 10.0 else "no motion"
        print(f"  {key}: {reason} ({measured:.2f}s, motion {motion:.4f})")
else:
    print(f"all {len(record['clips'])} clips are in the window and genuinely move")

## 8. Download

In [ ]:
import shutil

archive = shutil.make_archive("/content/research_clips", "zip", root_dir=WORK, base_dir="research")
size_mb = Path(archive).stat().st_size / 1e6
print(f"{archive}  ({size_mb:.1f} MB)")
files.download(archive)

## 9. On the laptop

```bash
cd fixtures && unzip -o ~/Downloads/research_clips.zip
.venv/bin/python -c "
import sys; sys.path.insert(0, 'packages/providers'); sys.path.insert(0, 'packages/schema')
from adproviders import ClipManifest
print(ClipManifest.load('fixtures/research').summary())"
```

Then generate the research-tier video corpus. No spend switch, no key:

```bash
AD_VIDEO_PROVIDER=ltx-video .venv/bin/python scripts/generate_corpus.py --sets 20 --videos
```

## What this notebook does not do

**Wan 2.1 chaining is not implemented here.** Wan caps at 5 s, so reaching the 8-10 s
window needs two generations concatenated, and the seam is a real quality cost. The
laptop can already *measure* that cost — `adml.video.detect_cuts` finds a seam from the
pixels alone, so a chained clip is detected whether or not the manifest admits to one —
but generating chained clips is week-14 work if the LTX-Video route proves insufficient.
Until then `wan-2.1-i2v` resolves to this same provider and will simply have no clips.

**No audio.** Veo 3.1's native audio is `$0.40/s` and out of reach. The music bed and TTS
voiceover are mixed locally with ffmpeg at the delivery stage.